# 06. Limpeza: notificações de dengue do Nordeste

Cria a tabela tratada `notificacao_dengue_ne` a partir da tabela bruta `sinan_dengue`, com as regras definidas a partir
do `05_analise.ipynb`. O trabalho é feito no próprio banco (SQL), porque são 5,6 milhões de linhas; aqui ficam as
regras, a criação e as conferências.

| Entrada | Saída |
|---|---|
| `sinan_dengue` (bruta, Brasil, 32,5 milhões) + `municipio` | `notificacao_dengue_ne`: uma linha por notificação de residente do Nordeste |

A execução padrão confere a tabela já existente. Em um banco vazio, cria a tabela automaticamente. Use `RECRIAR_TRATADA = True` para aplicar novamente a limpeza após atualizar a carga bruta.

**Neste notebook**
1. Regras
2. Criar a tabela
3. Conferência
4. Comparação histórica para Bahia em 2024

In [1]:
import sys
from pathlib import Path
RAIZ = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "config.py").exists())
sys.path.append(str(RAIZ / "src"))

import pandas as pd
import psycopg
from config import BANCO
from limpeza.dengue import limpeza_dengue

def consulta(q):
    with psycopg.connect(**BANCO) as con:
        cur = con.execute(q)
        return pd.DataFrame(cur.fetchall(), columns=[d.name for d in cur.description])

## 1. Regras

**Quem entra:** residentes do Nordeste, pelo município de residência (código que existe na tabela `municipio` do IBGE
e é de um estado de 21 a 29). Entram também os residentes notificados fora do Nordeste. Ficam de fora os códigos sem
par no IBGE (incluindo códigos que informam a UF sem identificar o município).

| Coluna | Regra |
|---|---|
| `cod_mun7` | código completo do IBGE do município de residência (o SINAN usa 7 dígitos até 2006 e 6 depois) |
| `cod_mun7_notificacao` | idem, do município que notificou |
| `ano` | ano da notificação |
| `dt_notificacao`, `dt_sintomas` | convertidas para data; inválidas viram vazio. Sintomas depois da notificação ou mais de 1 ano antes dela também |
| `mes_sintomas` | mês dos primeiros sintomas (para a sazonalidade) |
| `sexo` | M ou F; ignorado vira vazio |
| `idade_anos` | decodificada das duas fichas, tirando o `.0` que aparece em alguns anos; menos de 1 ano = 0; ignorada, sem unidade ou acima de 110 viram vazio |
| `classificacao`, `criterio`, `evolucao` | ficha antiga e nova num campo só |
| `caso_provavel` | exclui descartado (5) e chikungunya (13); inclui ausência de classificação e código 0 |
| `caso_confirmado` | códigos 1 a 4 (ficha até 2013, também usados em 2016) e 10 a 12 (desde 2014) |
| `possivel_duplicata` | marca a 2ª linha em diante de grupos iguais em notificação, datas, sexo, idade e residência (não apaga) |
| `ano_arquivo` | de qual CSV veio |

## 2. Preparar a tabela
Em um banco já carregado, esta etapa usa a tabela existente. Se a recriação estiver habilitada, a limpeza substitui a tabela em uma única transação.

In [2]:
RECRIAR_TRATADA = False
existe = consulta("SELECT to_regclass('notificacao_dengue_ne') IS NOT NULL AS existe").existe.iat[0]
if RECRIAR_TRATADA or not existe:
    linhas = limpeza_dengue.criar()
    print(f"notificacao_dengue_ne criada: {linhas:,} linhas")
else:
    print("Tabela tratada existente. As conferências abaixo usam o banco atual.")

Tabela tratada existente. As conferências abaixo usam o banco atual.


## 3. Conferência

### 3.1 Nenhuma notificação perdida
Residentes do Nordeste no bruto, menos os "município ignorado", deve ser igual ao total da tabela.

In [3]:
reconciliacao = consulta("""
    WITH bruto AS (
        SELECT count(*) AS residentes_ne_no_bruto,
               count(*) FILTER (WHERE m.cod_mun6 IS NULL) AS municipio_ignorado
        FROM sinan_dengue s
        LEFT JOIN municipio m ON m.cod_mun6 = left(s.id_mn_resi, 6)
                              AND m.uf_cod BETWEEN 21 AND 29
        WHERE left(s.id_mn_resi, 2) BETWEEN '21' AND '29'
    )
    SELECT bruto.*, (SELECT count(*) FROM notificacao_dengue_ne) AS na_tabela
    FROM bruto
""")
r = reconciliacao.iloc[0]
assert r.residentes_ne_no_bruto - r.municipio_ignorado == r.na_tabela, reconciliacao
reconciliacao

,residentes_ne_no_bruto,municipio_ignorado,na_tabela
0,5603997,167,5603830


### 3.2 Quanto cada regra afetou
Valores que ficaram vazios por causa das regras (ou que já vinham vazios).

In [4]:
consulta("""
    select count(*) as linhas,
           count(*) filter (where dt_notificacao is null) as dt_notificacao_vazia,
           count(*) filter (where dt_sintomas is null) as dt_sintomas_vazia,
           count(*) filter (where sexo is null) as sexo_vazio,
           count(*) filter (where idade_anos is null) as idade_vazia,
           count(*) filter (where classificacao is null) as classificacao_vazia,
           count(*) filter (where cod_mun7_notificacao is null) as municipio_notificacao_vazio
    from notificacao_dengue_ne""").T.rename(columns={0: "notificacoes"})

,notificacoes
linhas,5603830
dt_notificacao_vazia,1
dt_sintomas_vazia,13046
sexo_vazio,6777
idade_vazia,10829
classificacao_vazia,222428
municipio_notificacao_vazio,1


### 3.3 Casos por ano
Notificações, casos prováveis, casos confirmados e possíveis duplicadas por ano.

In [5]:
consulta("""
    select ano, count(*) as notificacoes,
           count(*) filter (where caso_provavel) as provaveis,
           count(*) filter (where caso_confirmado) as confirmados,
           count(*) filter (where possivel_duplicata) as possiveis_duplicatas,
           round(100.0 * count(*) filter (where possivel_duplicata) / count(*), 1) as pct_duplicatas
    from notificacao_dengue_ne group by ano order by ano""").set_index("ano")

,notificacoes,provaveis,confirmados,possiveis_duplicatas,pct_duplicatas
ano,,,,,
1999,1,1,1,0,0.0
2000,83283,70187,50929,1169,1.4
2001,180931,149734,95902,3475,1.9
2002,326105,267759,182581,8761,2.7
2003,207668,151541,109463,3516,1.7
2004,41850,22368,17025,288,0.7
2005,124662,76039,61463,3132,2.5
2006,107714,69757,59183,2255,2.1
2007,175517,125608,91581,2717,1.5


### 3.4 Idade e sexo
Distribuição da idade decodificada (deve ir de 0 a 110, com a maior parte entre 10 e 60) e do sexo.

In [6]:
consulta("""
    select min(idade_anos) as minima, percentile_cont(0.25) within group (order by idade_anos) as p25,
           percentile_cont(0.5) within group (order by idade_anos) as mediana,
           percentile_cont(0.75) within group (order by idade_anos) as p75, max(idade_anos) as maxima,
           round(100.0 * count(*) filter (where sexo = 'F') / count(sexo), 1) as pct_feminino
    from notificacao_dengue_ne""")

,minima,p25,mediana,p75,maxima,pct_feminino
0,0,16.0,27.0,42.0,110,56.8


## 4. Comparação histórica para Bahia em 2024

A referência de **232.623** casos prováveis foi atribuída à Sesab para o período de 31/12/2023 a 07/12/2024. A consulta abaixo usa o ano de notificação de 2024 na tabela local. Os recortes diferem, então a proximidade dos totais serve como comparação de ordem de grandeza.

[Notícia que cita a Sesab](https://www.correio24horas.com.br/minha-bahia/bahia-teve-quase-395-de-aumento-nos-casos-de-dengue-em-2024-1224)

In [7]:
REFERENCIA_BA_2024 = 232_623
nosso = consulta("""
    select count(*) filter (where caso_provavel) as n
    from notificacao_dengue_ne n join municipio m using (cod_mun7)
    where n.ano = 2024 and m.uf_cod = 29""")["n"][0]
print(f"Bahia 2024, casos prováveis: tabela {nosso:,} | referência histórica {REFERENCIA_BA_2024:,} | "
      f"diferença {nosso - REFERENCIA_BA_2024:+,} ({(nosso - REFERENCIA_BA_2024) / REFERENCIA_BA_2024:+.2%})")

Bahia 2024, casos prováveis: tabela 232,455 | referência histórica 232,623 | diferença -168 (-0.07%)
